In [1]:
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output
from datetime import datetime

plt.rcParams['font.family'] = 'IPAGothic'

# ==========================================
# 1. 基本設定とデータ読み込み
# ==========================================
db_path = "/data/cat.db"

def load_all_data():
    conn = sqlite3.connect(db_path)

    df_raw = pd.read_sql("SELECT * FROM raw_data ORDER BY timestamp", conn)

    try:
        df_labels = pd.read_sql("SELECT * FROM labels", conn)
    except:
        df_labels = pd.DataFrame()

    conn.close()

    if not df_raw.empty:
        unit = "s" if df_raw["timestamp"].iloc[0] < 1e11 else "ms"
        df_raw["datetime"] = pd.to_datetime(
            df_raw["timestamp"],
            unit=unit,
            utc=True
        ).dt.tz_convert('Asia/Tokyo')
        return df_raw, df_labels, unit

    return df_raw, df_labels, "s"


df, labels, ts_unit = load_all_data()

# ==========================================
# 2. UI
# ==========================================
plot_output = widgets.Output()
list_output = widgets.Output()

range_slider = widgets.SelectionRangeSlider(
    options=[('Loading...', 0)],
    index=(0, 0),
    description='Time Range',
    layout={'width': '95%'}
)

label_dropdown = widgets.Dropdown(
    options=[
        ('Pee (尿)', 'pee'),
        ('Poop (便)', 'poop'),
        ('Multiple (両方)', 'multiple'),
        ('Error (エラー)', 'error')
    ],
    value='pee',
    description='Tag:'
)

save_button = widgets.Button(description="SAVE TAG", button_style='success')
refresh_button = widgets.Button(description="REFRESH", button_style='info')

# ==========================================
# 3. スライダー更新
# ==========================================
def update_slider_options():
    global df, labels, ts_unit
    df, labels, ts_unit = load_all_data()

    if df.empty:
        return

    options = [
        (dt.strftime('%H:%M:%S'), float(ts))
        for dt, ts in zip(df["datetime"], df["timestamp"])
    ]

    range_slider.options = options
    range_slider.index = (max(0, len(options)-100), len(options)-1)


# ==========================================
# 4. 体重計算ロジック（直前1件・直後1件）
# ==========================================
def calculate_weight(start_ts, end_ts):

    before = df[df["timestamp"] < start_ts].sort_values("timestamp").tail(1)
    after = df[df["timestamp"] > end_ts].sort_values("timestamp").head(1)

    if before.empty or after.empty:
        return None, None, None

    entrance_weight = before["weight"].iloc[0]
    exit_weight = after["weight"].iloc[0]

    diff_weight = entrance_weight - exit_weight

    return entrance_weight, exit_weight, diff_weight


# ==========================================
# 5. 保存処理
# ==========================================
def on_save_clicked(b):

    start_ts, end_ts = range_slider.value
    label = label_dropdown.value

    entrance_w, exit_w, diff_w = calculate_weight(start_ts, end_ts)

    if diff_w is None:
        result_msg = "⚠️ データ不足で計算できません"
        return

    # 異常値フィルタ（1000g超は除外）
    if abs(diff_w) > 1000:
        result_msg = "⚠️ 異常値のため保存しません（1000g超）"
        with list_output:
            clear_output()
            display(widgets.HTML(f"<pre>{result_msg}</pre>"))
        return

    result_msg = (
        "📊【計算結果】\n"
        "-----------------------------\n"
        f"🐈 乗る直前の体重 : {entrance_w:.1f} g\n"
        f"🐈 降りた直後の体重 : {exit_w:.1f} g\n"
        f"🚽 排泄量         : {diff_w:.1f} g\n"
        "-----------------------------"
    )

    conn = sqlite3.connect(db_path)

    # カラム追加（存在しなければ）
    try:
        conn.execute("ALTER TABLE labels ADD COLUMN entrance_weight REAL")
    except:
        pass

    try:
        conn.execute("ALTER TABLE labels ADD COLUMN exit_weight REAL")
    except:
        pass

    try:
        conn.execute("ALTER TABLE labels ADD COLUMN diff_weight REAL")
    except:
        pass

    conn.execute("""
        INSERT INTO labels
        (start_ts, end_ts, label, entrance_weight, exit_weight, diff_weight)
        VALUES (?, ?, ?, ?, ?, ?)
    """, (start_ts, end_ts, label, entrance_w, exit_w, diff_w))

    conn.commit()
    conn.close()

    with list_output:
        clear_output()
        display(widgets.HTML(f"<pre>{result_msg}</pre>"))

    draw_graph()


# ==========================================
# 6. グラフ描画
# ==========================================
def draw_graph(change=None):

    with plot_output:
        clear_output(wait=True)

        if df.empty:
            print("No data found.")
            return

        s, e = range_slider.value

        fig, ax = plt.subplots(figsize=(15, 4))

        # 全体グラフ
        ax.plot(df["timestamp"], df["weight"], color='gray', alpha=0.3)

        # 選択範囲
        mask_sel = (df["timestamp"] >= s) & (df["timestamp"] <= e)
        selected_points = df.loc[mask_sel]
        ax.plot(selected_points["timestamp"], selected_points["weight"], 'r-', linewidth=2)

        # ===== タグ表示 =====
        if not labels.empty:
            for _, row in labels.iterrows():

                start = row["start_ts"]
                label_type = row["label"]

                if s <= start <= e:

                    if label_type == "pee":
                        text = "P"
                        color = "blue"
                    elif label_type == "poop":
                        text = "U"
                        color = "brown"
                    elif label_type == "error":
                        text = "E"
                        color = "red"
                    else:
                        continue

                    ax.axvline(start, linestyle="--", alpha=0.4, color=color)

                    ax.text(
                        start,
                        df["weight"].max(),
                        text,
                        fontsize=12,
                        ha='center',
                        va='bottom',
                        color=color
                    )

        def time_formatter(x, p):
            return pd.to_datetime(x, unit=ts_unit, utc=True)\
                     .tz_convert('Asia/Tokyo')\
                     .strftime('%H:%M:%S')

        ax.xaxis.set_major_formatter(plt.FuncFormatter(time_formatter))
        ax.set_xlim(s, e)
        ax.grid(True, alpha=0.2)

        plt.show()



# ==========================================
# 7. イベント登録
# ==========================================
range_slider.observe(draw_graph, names='value')
save_button.on_click(on_save_clicked)
refresh_button.on_click(lambda b: (update_slider_options(), draw_graph()))

# ==========================================
# 8. 実行
# ==========================================
update_slider_options()

display(widgets.HBox([refresh_button, label_dropdown, save_button]))
display(range_slider)
display(plot_output)
display(list_output)

draw_graph()


SelectionRangeSlider(description='Time Range', index=(25764, 25862), layout=Layout(width='95%'), options=(('10…

Output()

Output()